In [ ]:
# Configuration: where the dataset is written (see src/paths.py).
# Set PORES to 1, 2, 4 or 8; the dataset goes to THERMAL_GAN_DATA/simdata<PORES>
# (default: <repo>/data/simdata<PORES>), which is where the training script reads it.
import os, sys
from pathlib import Path
_here = Path.cwd()
_repo = next(p for p in [_here, *_here.parents] if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(_repo / "src"))
from paths import simdata_dir, output_dir

NUM_ELLIPSES = int(os.environ.get("PORES", 8))
SIM_DIR = str(simdata_dir(NUM_ELLIPSES))
print("Writing", NUM_ELLIPSES, "-pore dataset to", SIM_DIR)


In [ ]:
# One-sample thermal test with figures (mshr, no Gmsh)
import math, random, os
import numpy as np

import matplotlib
matplotlib.use("Agg")  # for headless runs; remove if you want interactive
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

from dolfin import *
from mshr import Rectangle, Polygon, generate_mesh
from PIL import Image, ImageDraw

# ----------------------------
# Controls
# ----------------------------
out_dir = str(output_dir("thermal_one_sample_out"))
os.makedirs(out_dir, exist_ok=True)

mesh_res   = 140
k_material = 1.0
num_ellipses = NUM_ELLIPSES
n_ellipse_pts = 64

minor_diam_range = (0.06, 0.14)
max_aspect = 3.0
margin = 0.02
max_tries = 8000

# ----------------------------
# Structure image (binary)
# ----------------------------
def draw_binary_image_ellipses(ellipses, size=128):
    img = Image.new("L", (size, size), "white")
    draw = ImageDraw.Draw(img)
    for (xc, yc, a, b, th) in ellipses:
        pts = []
        for i in range(n_ellipse_pts):
            ang = 2*math.pi*i/n_ellipse_pts
            x = a*math.cos(ang); y = b*math.sin(ang)
            xr =  x*math.cos(th) - y*math.sin(th)
            yr =  x*math.sin(th) + y*math.cos(th)
            px = (xc + xr) * size
            py = (1.0 - (yc + yr)) * size
            pts.append((px, py))
        draw.polygon(pts, fill="black")
    return np.array(img, dtype=np.uint8)

# ----------------------------
# Geometry helpers
# ----------------------------
def sample_identical_ellipses_nonoverlap(
    n=8,
    short_diam_range=(0.1, 0.3),   # short DIAMETER in domain units
    aspect_range=(2.0, 4.0),       # long/short
    theta_range=(0.0, 2*math.pi),  # orientation for the (identical) ellipses
    margin=0.02,                   # extra spacing and edge buffer
    max_tries=8000
):
    """
    Returns n tuples (xc, yc, a, b, theta) for identical ellipses:
      - short axis diameter ~ U[0.1, 0.3]
      - aspect ratio ~ U[2, 4]  (long/short)
      - same (a,b,theta) for all 8; centers differ
      - non-overlapping via conservative circle bound (R = max(a,b) + margin)
      - auto-scales (a,b) down if rotated footprint wouldn't fit in [0,1]^2
    """

    # --- 1) Choose one ellipse geometry for this sample ---
    short_d = random.uniform(*short_diam_range)    # short DIAMETER
    aspect  = random.uniform(*aspect_range)        # >= 1
    long_d  = aspect * short_d

    a = long_d / 2.0   # semi-major
    b = short_d / 2.0  # semi-minor
    theta = random.uniform(*theta_range)

    # Compute rotated half-width/half-height footprint
    def footprint(a_, b_, th):
        xm_ = abs(a_*math.cos(th)) + abs(b_*math.sin(th))
        ym_ = abs(a_*math.sin(th)) + abs(b_*math.cos(th))
        return xm_, ym_

    xm, ym = footprint(a, b, theta)

    # If it doesn't fit, scale it down uniformly to fit (with margin)
    # Condition: xm + margin < 0.5 and ym + margin < 0.5
    # Find scale s <= 1 so s*xm <= 0.5 - margin and s*ym <= 0.5 - margin
    cap_x = 0.5 - margin
    cap_y = 0.5 - margin
    if xm > cap_x or ym > cap_y:
        sx = cap_x / max(xm, 1e-12)
        sy = cap_y / max(ym, 1e-12)
        s = max(min(sx, sy), 0.0)
        s *= 0.98  # small cushion
        a *= s
        b *= s
        xm, ym = footprint(a, b, theta)

    # --- 2) Non-overlap using conservative circle bound ---
    R = max(a, b) + margin
    min_center_dist2 = (2.0 * R) ** 2

    # Allowed placement box after footprint + margin
    x_lo, x_hi = xm + margin, 1.0 - (xm + margin)
    y_lo, y_hi = ym + margin, 1.0 - (ym + margin)

    # If still degenerate (can happen for very large short_d), relax once
    if not (x_lo < x_hi and y_lo < y_hi):
        # fallback: shrink ellipse a bit and recompute
        shrink = 0.9
        a *= shrink; b *= shrink
        xm, ym = footprint(a, b, theta)
        R = max(a, b) + margin
        min_center_dist2 = (2.0 * R) ** 2
        x_lo, x_hi = xm + margin, 1.0 - (xm + margin)
        y_lo, y_hi = ym + margin, 1.0 - (ym + margin)

        # final guard
        if not (x_lo < x_hi and y_lo < y_hi):
            # last-resort safety: minimal placement box
            eps = 1e-3
            x_lo, x_hi = eps, 1.0 - eps
            y_lo, y_hi = eps, 1.0 - eps

    centers = []
    tries = 0
    while len(centers) < n and tries < max_tries:
        tries += 1
        xc = random.uniform(x_lo, x_hi)
        yc = random.uniform(y_lo, y_hi)
        if all((xc - x2)**2 + (yc - y2)**2 >= min_center_dist2 for (x2, y2) in centers):
            centers.append((xc, yc))

    if len(centers) < n:
        # If packing failed (ellipse too big or margin too strict), relax margin once and retry
        return sample_identical_ellipses_nonoverlap(
            n=n,
            short_diam_range=short_diam_range,
            aspect_range=aspect_range,
            theta_range=theta_range,
            margin=max(0.0, 0.5 * margin),
            max_tries=max_tries
        )

    return [(xc, yc, a, b, theta) for (xc, yc) in centers]


def build_domain_with_holes(ellipses):
    rect = Rectangle(Point(0.0, 0.0), Point(1.0, 1.0))
    domain = rect
    for (xc, yc, a, b, th) in ellipses:
        pts = []
        for i in range(n_ellipse_pts):
            ang = 2*math.pi*i/n_ellipse_pts
            x = a*math.cos(ang); y = b*math.sin(ang)
            xr =  x*math.cos(th) - y*math.sin(th)
            yr =  x*math.sin(th) + y*math.cos(th)
            pts.append(Point(xc + xr, yc + yr))
        domain = domain - Polygon(pts)
    return domain

def mark_boundaries(mesh):
    boundaries = MeshFunction("size_t", mesh, mesh.topology().dim()-1, 0)
    class Left(SubDomain):
        def inside(self, x, on_b): return on_b and near(x[0], 0.0)
    class Right(SubDomain):
        def inside(self, x, on_b): return on_b and near(x[0], 1.0)
    class Bottom(SubDomain):
        def inside(self, x, on_b): return on_b and near(x[1], 0.0)
    class Top(SubDomain):
        def inside(self, x, on_b): return on_b and near(x[1], 1.0)
    Left().mark(boundaries, 1)
    Right().mark(boundaries, 2)
    Bottom().mark(boundaries, 3)
    Top().mark(boundaries, 4)
    return boundaries

# ----------------------------
# Solvers + plotting helpers
# ----------------------------
def solve_T_and_keff_x(mesh, kappa, boundaries):
    V = FunctionSpace(mesh, "CG", 1)
    u = TrialFunction(V); v = TestFunction(V)
    bc_left  = DirichletBC(V, Constant(0.0), boundaries, 1)
    bc_right = DirichletBC(V, Constant(1.0), boundaries, 2)
    a = inner(kappa*grad(u), grad(v))*dx
    L = Constant(0.0)*v*dx
    T = Function(V)
    solve(a == L, T, [bc_left, bc_right])
    n = FacetNormal(mesh)
    ds_b = Measure("ds", domain=mesh, subdomain_data=boundaries)
    flux_left = assemble(-kappa*dot(grad(T), n) * ds_b(1))  # positive
    return T, float(flux_left)

def solve_T_and_keff_y(mesh, kappa, boundaries):
    V = FunctionSpace(mesh, "CG", 1)
    u = TrialFunction(V); v = TestFunction(V)
    bc_bottom = DirichletBC(V, Constant(0.0), boundaries, 3)
    bc_top    = DirichletBC(V, Constant(1.0), boundaries, 4)
    a = inner(kappa*grad(u), grad(v))*dx
    L = Constant(0.0)*v*dx
    T = Function(V)
    solve(a == L, T, [bc_bottom, bc_top])
    n = FacetNormal(mesh)
    ds_b = Measure("ds", domain=mesh, subdomain_data=boundaries)
    flux_bottom = assemble(-kappa*dot(grad(T), n) * ds_b(3))  # positive
    return T, float(flux_bottom)

def save_mesh_png(mesh, path):
    coords = mesh.coordinates()
    cells = mesh.cells()
    tri = mtri.Triangulation(coords[:,0], coords[:,1], cells)
    plt.figure(figsize=(5,5))
    plt.triplot(tri, linewidth=0.4)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.title("Mesh")
    plt.xlabel("x"); plt.ylabel("y")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

def save_field_png(mesh, field, path, title):
    coords = mesh.coordinates()
    cells = mesh.cells()
    tri = mtri.Triangulation(coords[:,0], coords[:,1], cells)
    vals = field.compute_vertex_values(mesh)
    plt.figure(figsize=(5,5))
    im = plt.tripcolor(tri, vals, shading="gouraud")
    plt.gca().set_aspect('equal', adjustable='box')
    plt.colorbar(im, label="Temperature")
    plt.title(title)
    plt.xlabel("x"); plt.ylabel("y")
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()

# ----------------------------
# Build ONE sample
# ----------------------------
ellipses = sample_identical_ellipses_nonoverlap(
    n=8,
    short_diam_range=(0.05, 0.2),   # short DIAMETER ∈ [0.1, 0.3]
    aspect_range=(2.0, 6.0),       # aspect ∈ [2, 4]
    theta_range=(0.0, 2*math.pi),  # keep identical random orientation per sample
    margin=0.02,
    max_tries=8000
)

# Save structure image
# struct_img = draw_binary_image_ellipses(ellipses, size=128)  # higher res if you like
# Make binary image in [0,255] for saving as PNG
struct_img_uint8 = draw_binary_image_ellipses(ellipses, size=128)

# Save PNG (will be black/white image)
Image.fromarray(struct_img_uint8).save(os.path.join(out_dir, "structure.png"))

# Normalize to [0,1] and save as .npy for ML training
struct_img_float = struct_img_uint8.astype(np.float32) / 255.0
np.save(os.path.join(out_dir, "structure.npy"), struct_img_float)


# Build domain + mesh
domain = build_domain_with_holes(ellipses)
mesh = generate_mesh(domain, mesh_res)
save_mesh_png(mesh, os.path.join(out_dir, "mesh.png"))

# Boundary markers and material
boundaries = mark_boundaries(mesh)
kappa = Constant(k_material)

# Solve X case
T_x, kx = solve_T_and_keff_x(mesh, kappa, boundaries)
save_field_png(mesh, T_x, os.path.join(out_dir, "T_x.png"), "Temperature (x-load)")
File(os.path.join(out_dir, "T_x.pvd")) << T_x  # optional for ParaView

# Solve Y case
T_y, ky = solve_T_and_keff_y(mesh, kappa, boundaries)
save_field_png(mesh, T_y, os.path.join(out_dir, "T_y.png"), "Temperature (y-load)")
File(os.path.join(out_dir, "T_y.pvd")) << T_y  # optional for ParaView

# Report
print(f"kx = {kx:.6f}")
print(f"ky = {ky:.6f}")
print(f"Saved: {os.path.abspath(out_dir)}")


In [ ]:
# ============================================================================
# Thermal Dataset Generator (mshr, no Gmsh) — 4000 samples
#   - 8 identical, non-overlapping ellipses (holes) per sample
#   - aspect ratio ~ U[2, 4], short diameter ~ U[0.1, 0.3]
#   - saves: struct_####.png, struct_####.npy, thermal_####.npz (kx, ky, ellipses)
# ============================================================================
import os, math, random
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw

import matplotlib
matplotlib.use("Agg")  # headless-safe
import matplotlib.pyplot as plt

from dolfin import *
from mshr import Rectangle, Polygon, generate_mesh

# ----------------------------------------------------------------------------
# Config
# ----------------------------------------------------------------------------
sim_dir = SIM_DIR
os.makedirs(sim_dir, exist_ok=True)

num_samples      = 10000
start_sample_idx = 0

img_size   = 128      # structure image resolution
mesh_res   = 140      # FEM mesh resolution
k_material = 1.0      # conductivity of solid; holes are removed from domain

num_ellipses   = NUM_ELLIPSES
n_ellipse_pts  = 64

# ellipse distributions
short_diam_range = (0.05, 0.2)   # DIAMETER of short axis
aspect_range     = (2.0, 6.0)   # long/short
theta_range      = (0.0, 2*math.pi)

# packing / geometry
margin     = 0.02
max_tries  = 8000

# ----------------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------------
def is_valid_npz(path):
    try:
        data = np.load(path)
        for k in ("kx", "ky"):
            if k not in data: return False
            v = float(data[k])
            if not np.isfinite(v) or v <= 0 or v > 1e6: return False
        return True
    except Exception:
        return False

def draw_binary_image_ellipses(ellipses, size=128):
    """Return uint8 image (255=solid, 0=hole)"""
    img = Image.new("L", (size, size), "white")
    draw = ImageDraw.Draw(img)
    for (xc, yc, a, b, th) in ellipses:
        pts = []
        for i in range(n_ellipse_pts):
            ang = 2*math.pi*i/n_ellipse_pts
            x = a*math.cos(ang); y = b*math.sin(ang)
            xr =  x*math.cos(th) - y*math.sin(th)
            yr =  x*math.sin(th) + y*math.cos(th)
            px = (xc + xr) * size
            py = (1.0 - (yc + yr)) * size  # flip y for image coords
            pts.append((px, py))
        draw.polygon(pts, fill="black")
    return np.array(img, dtype=np.uint8)

def sample_identical_ellipses_nonoverlap(
    n=8, short_diam_range=(0.1,0.3), aspect_range=(2.0,4.0),
    theta_range=(0.0, 2*math.pi), margin=0.02, max_tries=8000
):
    """Identical (a,b,theta), non-overlapping centers via conservative circle bound."""
    # one geometry for all
    short_d = random.uniform(*short_diam_range)   # short DIAMETER
    aspect  = random.uniform(*aspect_range)       # long/short >=1
    long_d  = aspect * short_d
    a = long_d/2.0
    b = short_d/2.0
    theta = random.uniform(*theta_range)

    # rotated half-footprint
    def footprint(a_, b_, th):
        xm_ = abs(a_*math.cos(th)) + abs(b_*math.sin(th))
        ym_ = abs(a_*math.sin(th)) + abs(b_*math.cos(th))
        return xm_, ym_

    xm, ym = footprint(a, b, theta)
    cap = 0.5 - margin
    if xm > cap or ym > cap:  # scale down uniformly to fit
        s = min(cap/max(xm,1e-12), cap/max(ym,1e-12)) * 0.98
        a *= s; b *= s
        xm, ym = footprint(a, b, theta)

    # conservative circle radius for spacing
    R = max(a, b) + margin
    min_center_d2 = (2.0*R)**2

    # allowed placement box
    x_lo, x_hi = xm + margin, 1.0 - (xm + margin)
    y_lo, y_hi = ym + margin, 1.0 - (ym + margin)
    if not (x_lo < x_hi and y_lo < y_hi):
        # shrink slightly and recompute once
        a *= 0.9; b *= 0.9
        xm, ym = footprint(a, b, theta)
        R = max(a, b) + margin
        min_center_d2 = (2.0*R)**2
        x_lo, x_hi = xm + margin, 1.0 - (xm + margin)
        y_lo, y_hi = ym + margin, 1.0 - (ym + margin)

    centers, tries = [], 0
    while len(centers) < n and tries < max_tries:
        tries += 1
        xc = random.uniform(x_lo, x_hi)
        yc = random.uniform(y_lo, y_hi)
        if all((xc-x2)**2 + (yc-y2)**2 >= min_center_d2 for (x2,y2) in centers):
            centers.append((xc, yc))

    if len(centers) < n:
        # relax margin once
        return sample_identical_ellipses_nonoverlap(
            n, short_diam_range, aspect_range, theta_range, max(0.5*margin, 0.0), max_tries
        )

    return [(xc, yc, a, b, theta) for (xc, yc) in centers]

def build_domain_with_holes(ellipses):
    rect = Rectangle(Point(0.0, 0.0), Point(1.0, 1.0))
    domain = rect
    for (xc, yc, a, b, th) in ellipses:
        pts = []
        for i in range(n_ellipse_pts):
            ang = 2*math.pi*i/n_ellipse_pts
            x = a*math.cos(ang); y = b*math.sin(ang)
            xr =  x*math.cos(th) - y*math.sin(th)
            yr =  x*math.sin(th) + y*math.cos(th)
            pts.append(Point(xc + xr, yc + yr))
        domain = domain - Polygon(pts)
    return domain

def mark_boundaries(mesh):
    boundaries = MeshFunction("size_t", mesh, mesh.topology().dim()-1, 0)
    class Left(SubDomain):
        def inside(self, x, on_b): return on_b and near(x[0], 0.0)
    class Right(SubDomain):
        def inside(self, x, on_b): return on_b and near(x[0], 1.0)
    class Bottom(SubDomain):
        def inside(self, x, on_b): return on_b and near(x[1], 0.0)
    class Top(SubDomain):
        def inside(self, x, on_b): return on_b and near(x[1], 1.0)
    Left().mark(boundaries, 1)
    Right().mark(boundaries, 2)
    Bottom().mark(boundaries, 3)
    Top().mark(boundaries, 4)
    return boundaries

def solve_T_and_keff_x(mesh, kappa, boundaries):
    V = FunctionSpace(mesh, "CG", 1)
    u = TrialFunction(V); v = TestFunction(V)
    bc_left  = DirichletBC(V, Constant(0.0), boundaries, 1)
    bc_right = DirichletBC(V, Constant(1.0), boundaries, 2)
    a = inner(kappa*grad(u), grad(v))*dx
    L = Constant(0.0)*v*dx
    T = Function(V)
    solve(a == L, T, [bc_left, bc_right])
    n = FacetNormal(mesh)
    ds_b = Measure("ds", domain=mesh, subdomain_data=boundaries)
    flux_left = assemble(-kappa*dot(grad(T), n) * ds_b(1))  # positive
    return T, float(flux_left)  # ΔT=1

def solve_T_and_keff_y(mesh, kappa, boundaries):
    V = FunctionSpace(mesh, "CG", 1)
    u = TrialFunction(V); v = TestFunction(V)
    bc_bottom = DirichletBC(V, Constant(0.0), boundaries, 3)
    bc_top    = DirichletBC(V, Constant(1.0), boundaries, 4)
    a = inner(kappa*grad(u), grad(v))*dx
    L = Constant(0.0)*v*dx
    T = Function(V)
    solve(a == L, T, [bc_bottom, bc_top])
    n = FacetNormal(mesh)
    ds_b = Measure("ds", domain=mesh, subdomain_data=boundaries)
    flux_bottom = assemble(-kappa*dot(grad(T), n) * ds_b(3))  # positive
    return T, float(flux_bottom)  # ΔT=1

# ----------------------------------------------------------------------------
# Main generation loop
# ----------------------------------------------------------------------------
generated = 0
while generated < num_samples:
    sample_idx = start_sample_idx + generated

    npz_path = os.path.join(sim_dir, f"thermal_{sample_idx:04d}.npz")
    png_path = os.path.join(sim_dir, f"struct_{sample_idx:04d}.png")
    npy_path = os.path.join(sim_dir, f"struct_{sample_idx:04d}.npy")

    # Skip if already valid
    if Path(npz_path).exists() and is_valid_npz(npz_path) and Path(png_path).exists() and Path(npy_path).exists():
        print(f"Skipping existing valid sample {sample_idx}")
        generated += 1
        continue

    print(f"Generating thermal sample {sample_idx}")
    try:
        # --- 1) Geometry: 8 identical, non-overlapping ellipses ---
        ellipses = sample_identical_ellipses_nonoverlap(
            n=num_ellipses,
            short_diam_range=short_diam_range,
            aspect_range=aspect_range,
            theta_range=theta_range,
            margin=margin,
            max_tries=max_tries
        )

        # --- 2) Save structure image (PNG + NPY normalized) ---
        struct_uint8 = draw_binary_image_ellipses(ellipses, size=img_size)  # [0,255]
        Image.fromarray(struct_uint8).save(png_path)
        struct_float = (struct_uint8.astype(np.float32) / 255.0)           # [0,1]
        np.save(npy_path, struct_float)

        # --- 3) Mesh & BCs ---
        domain = build_domain_with_holes(ellipses)
        mesh = generate_mesh(domain, mesh_res)
        boundaries = mark_boundaries(mesh)
        kappa = Constant(k_material)

        # --- 4) Solve for kx, ky ---
        _, kx = solve_T_and_keff_x(mesh, kappa, boundaries)
        _, ky = solve_T_and_keff_y(mesh, kappa, boundaries)

        # --- 5) Save scalars + geometry params ---
        np.savez(npz_path,
                 kx=kx, ky=ky,
                 k_material=k_material,
                 num_ellipses=num_ellipses,
                 ellipses=np.array(ellipses, dtype=float))  # (xc, yc, a, b, theta)

        # Final validity check
        if not is_valid_npz(npz_path):
            print(f"Invalid data at sample {sample_idx}; deleting file.")
            try:
                os.remove(npz_path)
            except OSError:
                pass
            # keep images for debugging; do not increment
            continue

        generated += 1

    except Exception as e:
        print(f"Exception during sample {sample_idx}: {e}")
        # try this index again
        continue

print("Done.")


In [ ]:
import os, random, sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

# If you're in a headless environment (e.g., script), use a non-interactive backend
# matplotlib.use("Agg")  # uncomment if needed

sim_dir = SIM_DIR

if not os.path.isdir(sim_dir):
    print(f"[ERROR] Directory not found: {sim_dir}")
    sys.exit(0)

# Gather indices that have BOTH thermal_{id}.npz and struct_{id}.png
npz_ids = {int(f.split("_")[1].split(".")[0]) for f in os.listdir(sim_dir)
           if f.startswith("thermal_") and f.endswith(".npz")}
png_ids = {int(f.split("_")[1].split(".")[0]) for f in os.listdir(sim_dir)
           if f.startswith("struct_") and f.endswith(".png")}
ids = sorted(npz_ids & png_ids)

print(f"[INFO] Found {len(npz_ids)} .npz files, {len(png_ids)} PNGs, {len(ids)} with both.")

if len(ids) == 0:
    print("[WARN] No matching pairs found. Make sure you ran the generator and paths match.")
    # Optional: show a few filenames to debug
    print("Some files in dir:", sorted(os.listdir(sim_dir))[:10])
else:
    k = min(10, len(ids))
    sample_ids = random.sample(ids, k)
    print(f"[INFO] Sampling IDs: {sample_ids}")

    # Make the figure
    rows = 2
    cols = (k + rows - 1) // rows
    fig, axes = plt.subplots(rows, cols, figsize=(3.2*cols, 3.6*rows))
    axes = np.atleast_2d(axes)
    axes = axes.ravel()

    for ax, idx in zip(axes, sample_ids):
        # load kx, ky
        data = np.load(os.path.join(sim_dir, f"thermal_{idx:04d}.npz"))
        kx, ky = float(data["kx"]), float(data["ky"])

        # load image (use PNG for quick preview)
        img_path = os.path.join(sim_dir, f"struct_{idx:04d}.png")
        if not os.path.exists(img_path):
            # fallback to .npy if png missing
            npy_path = os.path.join(sim_dir, f"struct_{idx:04d}.npy")
            if os.path.exists(npy_path):
                img = (np.load(npy_path) * 255).astype(np.uint8)
            else:
                ax.text(0.5, 0.5, "missing image", ha="center", va="center")
                ax.axis("off")
                continue
        else:
            img = Image.open(img_path)

        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"id={idx}\nkx={kx:.3f}, ky={ky:.3f}", fontsize=9)
        ax.axis("off")

    # Hide any extra axes if k < rows*cols
    for j in range(k, rows*cols):
        axes[j].axis("off")

    plt.tight_layout()
    # Save collage so you can open even if plt.show() does nothing
    out_path = os.path.join(sim_dir, "sample_preview.png")
    plt.savefig(out_path, dpi=180)
    print(f"[INFO] Saved preview collage to: {out_path}")

    # Show if interactive
    try:
        plt.show()
    except Exception as e:
        print(f"[INFO] plt.show() not available: {e}")


In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt

# Point to your dataset folder
sim_dir = SIM_DIR

# Collect all .npz files
npz_files = sorted(glob.glob(os.path.join(sim_dir, "thermal_*.npz")))
if not npz_files:
    raise FileNotFoundError(f"No thermal_*.npz files found in {sim_dir}")

# Load kx, ky
kx_list, ky_list = [], []
for f in npz_files:
    d = np.load(f)
    kx_list.append(float(d["kx"]))
    ky_list.append(float(d["ky"]))

kx = np.array(kx_list)
ky = np.array(ky_list)

print(f"Loaded {len(kx)} samples")
print(f"kx: min={kx.min():.4f}, mean={kx.mean():.4f}, median={np.median(kx):.4f}, max={kx.max():.4f}, std={kx.std():.4f}")
print(f"ky: min={ky.min():.4f}, mean={ky.mean():.4f}, median={np.median(ky):.4f}, max={ky.max():.4f}, std={ky.std():.4f}")
# simple correlation for sanity
if len(kx) > 1:
    corr = np.corrcoef(kx, ky)[0,1]
    print(f"corr(kx, ky) = {corr:.4f}")

# 2D histogram (kx vs ky)
plt.figure(figsize=(6,5))
h = plt.hist2d(kx, ky, bins=30)  # default colormap; counts shown via colorbar
plt.xlabel("kx")
plt.ylabel("ky")
cb = plt.colorbar()
cb.set_label("count")
plt.tight_layout()
out2d = os.path.join(sim_dir, "hist2d_kx_ky.png")
plt.savefig(out2d, dpi=200)
plt.show()
print(f"Saved 2D histogram to: {out2d}")

# 1D histogram of kx
plt.figure(figsize=(6,4))
plt.hist(kx, bins=30)
plt.xlabel("kx"); plt.ylabel("count")
plt.tight_layout()
outx = os.path.join(sim_dir, "hist_kx.png")
plt.savefig(outx, dpi=200)
plt.show()
print(f"Saved kx histogram to: {outx}")

# 1D histogram of ky
plt.figure(figsize=(6,4))
plt.hist(ky, bins=30)
plt.xlabel("ky"); plt.ylabel("count")
plt.tight_layout()
outy = os.path.join(sim_dir, "hist_ky.png")
plt.savefig(outy, dpi=200)
plt.show()
print(f"Saved ky histogram to: {outy}")


In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt

# Folder with your npz files
sim_dir = SIM_DIR

# Load kx, ky from all samples
npz_files = sorted(glob.glob(os.path.join(sim_dir, "thermal_*.npz")))
kx, ky = [], []
for f in npz_files:
    d = np.load(f)
    kx.append(float(d["kx"]))
    ky.append(float(d["ky"]))
kx = np.array(kx); ky = np.array(ky)

# --- Build binned “circle histogram” (bubble plot) ---
bins = 30  # adjust as you like (e.g., 20, 40)
H, xedges, yedges = np.histogram2d(kx, ky, bins=bins)

# Bin centers
xc = 0.5 * (xedges[:-1] + xedges[1:])
yc = 0.5 * (yedges[:-1] + yedges[1:])
XX, YY = np.meshgrid(xc, yc, indexing="xy")

# Flatten and keep only non-empty bins
counts = H.T.ravel()          # transpose so X/Y align with meshgrid
cx = XX.ravel()
cy = YY.ravel()
mask = counts > 0
counts = counts[mask]
cx = cx[mask]
cy = cy[mask]

# Circle sizes (matplotlib 's' is area in points^2)
# Scale areas smoothly based on count
if len(counts) > 0:
    cmax = counts.max()
    s_min, s_max = 30.0, 800.0         # min/max bubble area (pts^2)
    sizes = s_min + (s_max - s_min) * (counts / cmax)
else:
    sizes = []

plt.figure(figsize=(6,5))
sc = plt.scatter(cx, cy, s=sizes, c=counts, cmap="viridis", alpha=0.85, edgecolors="none")
plt.xlabel("kx")
plt.ylabel("ky")
plt.title("2D Circle Histogram (kx vs ky)")
cb = plt.colorbar(sc); cb.set_label("count")
plt.tight_layout()

out_path = os.path.join(sim_dir, "hist2d_kx_ky_circles.png")
plt.savefig(out_path, dpi=200)
plt.show()
print(f"Saved circle-hist to: {out_path}")


In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(kx, ky, s=12, alpha=0.5)  # circles by default
plt.xlabel("kx"); plt.ylabel("ky")
plt.title("Raw (kx, ky) scatter")
plt.tight_layout()
plt.savefig(os.path.join(sim_dir, "scatter_kx_ky.png"), dpi=200)
plt.show()
